# 🔍 RAG Lab — Exploration Notebook
**Day 9 · AI Application Development Bootcamp**

This notebook walks you through the RAG pipeline step by step, letting you **see the output at every stage** before writing a full application.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Cells marked `# 🔬 EXPERIMENT` invite you to change values and observe what happens
- Write your reflections in the **📝 Reflection** markdown cells

### Sections at a glance
| Part | Topic | Type |
|---|---|---|
| A | Guided pipeline walk-through | Run & read |
| B | Parameter experiments | Fill in + reflect |
| C | Open exploration (your own PDF, multi-query, scores) | Open |
| D | **Web-Augmented RAG** — live web search as a retriever | Fill in |
| E | **RAG Evaluation with RAGAS** *(optional)* — measure quality | Fill in |

### Before you start
Make sure your virtual environment is active and `GROQ_API_KEY` is set:
```bash
source .venv/bin/activate          # macOS/Linux
# or
.venv\Scripts\Activate.ps1        # Windows PowerShell

export GROQ_API_KEY="gsk_..."
```
Then launch Jupyter:
```bash
pip install jupyter  # if not already installed
jupyter notebook
```

---
## ⚙️ Setup — Install & Import

In [6]:
# Run this cell first — installs everything needed for the notebook
# (skip if you already ran pip install during the lecture setup)
%pip install groq python-dotenv langchain langchain-community langchain-chroma langchain-huggingface chromadb pypdf sentence-transformers langchain-groq
print("✅ All packages ready")

Note: you may need to restart the kernel to use updated packages.
✅ All packages ready


In [7]:
import os
import shutil
import math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env if present

# Quick check — will print ✅ if the key is available
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print(f"✅ GROQ_API_KEY found (starts with: {api_key[:8]}...)")
else:
    print("❌ GROQ_API_KEY not found — set it before continuing")
    print("   export GROQ_API_KEY='gsk_your_key_here'   # macOS/Linux")
    print("   $env:GROQ_API_KEY='gsk_your_key_here'    # Windows PS")

✅ GROQ_API_KEY found (starts with: gsk_IfqM...)


---
## 🅐 PART A — Guided Pipeline Walk-Through

We will build the RAG pipeline one stage at a time and inspect the output at each step.  
Place any PDF you have in the same folder as this notebook and update `PDF_PATH` below.  
If you don't have one handy, you can save any Wikipedia article as a PDF from your browser.

### A1 — Step 1: Load the PDF

In [8]:
from langchain_community.document_loaders import PyPDFLoader

# 📌 Change this to your PDF file name
PDF_PATH = "sample.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"📄 Pages loaded: {len(pages)}")
print()
print("--- Metadata of the first page ---")
print(pages[0].metadata)
print()
print("--- First 500 characters of page 1 ---")
print(pages[0].page_content[:500])

📄 Pages loaded: 19

--- Metadata of the first page ---
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2021-02-09T12:15:53+08:00', 'title': 'Lecture 11', 'author': 'Claude Comair', 'subject': 'CSD1130            Game Implementation Techniques', 'moddate': '2021-02-09T12:15:53+08:00', 'source': 'sample.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}

--- First 500 characters of page 1 ---
Lecture 11 
Binary Collision 
Hot Spots 
1. Platformer – Binary Collision Map 2 
1.1. Introduction 2 
1.2. Initialization 3 
1.3. Point collision 4 
1.4. Sprite collision using hot spots 4 
1.5. Snapping 7 
1.6. Normalized coordinates system 8 
1.7. Flipping the Y value 11 
1.8. Recapitulation 13 
 
CSD1130            
Game 
Implementation 
Techniques


**What you should see:**
- `pages` is a Python list — one `Document` object per page
- Each Document has a `metadata` dict containing at minimum `page` (0-indexed) and `source` (the filename)
- These metadata fields travel with each chunk all the way to ChromaDB and show up in your citations

> **Why does the page number start at 0?** PyPDFLoader uses 0-based indexing internally. When displaying to users, add 1: `page + 1`.

### A2 — Step 2: Split into Chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(pages)

print(f"✂️  Total chunks created: {len(chunks)}")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print()
print("--- Sample: chunk #3 ---")
print(f"Metadata : {chunks[2].metadata}")
print(f"Length   : {len(chunks[2].page_content)} chars")
print(f"Content  : {chunks[2].page_content[:400]}")

✂️  Total chunks created: 37
   Average chunk size: 624 chars

--- Sample: chunk #3 ---
Metadata : {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2021-02-09T12:15:53+08:00', 'title': 'Lecture 11', 'author': 'Claude Comair', 'subject': 'CSD1130            Game Implementation Techniques', 'moddate': '2021-02-09T12:15:53+08:00', 'source': 'sample.pdf', 'total_pages': 19, 'page': 1, 'page_label': '2'}
Length   : 778 chars
Content  : game objects can only maneuver in “usable” areas. 
o The same logic applies in 3D games, the playe r has always some kind of restrictions 
concerning moving and controlling game objects. 
 
• Example: Platform games. 
o Movement of the main character is restricted: It can only walk on platforms. 
o Each level is divided into 2 main parts: Platforms and non-platforms (air). 
o World collision is “s


**Notice:**
- Chunk metadata still carries the original page number and source filename — inherited from the parent Document
- Chunk size varies because `RecursiveCharacterTextSplitter` respects natural break points
- `chunk_size=800` is the **maximum** — most chunks will be somewhat shorter

### A3 — Step 3: Load the Embedding Model

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading embedding model... (first run downloads ~80 MB)")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Model loaded")

# Embed a single test sentence and inspect the vector
test_vector = embeddings.embed_query("What is the deadline for submission?")

print(f"\n🔢 Vector dimensions : {len(test_vector)}")
print(f"   First 8 values    : {[round(v, 4) for v in test_vector[:8]]}")
print(f"   Min value         : {min(test_vector):.4f}")
print(f"   Max value         : {max(test_vector):.4f}")

Loading embedding model... (first run downloads ~80 MB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded

🔢 Vector dimensions : 384
   First 8 values    : [-0.0141, -0.0333, 0.0125, 0.0354, 0.0086, -0.0128, -0.1207, -0.0198]
   Min value         : -0.1540
   Max value         : 0.1974


**What you should see:**
- Vector has exactly **384 dimensions** (that's the output size of MiniLM-L6)
- Values are small floats, both positive and negative
- These 384 numbers together encode the *meaning* of the sentence

### A4 — Step 4: Demonstrate Cosine Similarity

Before storing anything, let's see cosine similarity in action — the mechanism that powers retrieval.

In [11]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot   = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x ** 2 for x in a))
    mag_b = math.sqrt(sum(x ** 2 for x in b))
    return dot / (mag_a * mag_b)

sentences = {
    "query"    : "What is the late submission policy?",
    "similar"  : "Penalty for assignments handed in after the deadline",
    "related"  : "Students must attend all classes",
    "unrelated": "The best recipe for banana bread",
}

vectors   = {k: embeddings.embed_query(v) for k, v in sentences.items()}
query_vec = vectors["query"]

print(f'Base query: "{sentences["query"]}"\n')
for key in ["similar", "related", "unrelated"]:
    score = cosine_similarity(query_vec, vectors[key])
    bar   = "█" * int(score * 30)
    print(f"{score:.4f}  {bar}")
    print(f'         → "{sentences[key]}"')
    print()

Base query: "What is the late submission policy?"

0.4022  ████████████
         → "Penalty for assignments handed in after the deadline"

0.0741  ██
         → "Students must attend all classes"

-0.0138  
         → "The best recipe for banana bread"



**Expected pattern:** `similar` should score highest (different words, same meaning), `related` moderate, `unrelated` very low.  
This is exactly what ChromaDB does for every query — computed across all stored chunk vectors simultaneously.

### A5 — Step 5: Store in ChromaDB

In [12]:
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_lab_notebook"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)
    print("🗑  Cleared old index")

print(f"Embedding {len(chunks)} chunks and storing in ChromaDB...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="lab_exploration",
)

count = vector_store._collection.count()
print(f"✅ Done — {count} vectors stored in ChromaDB")
print(f"   Index saved to: {CHROMA_DIR}/")

Embedding 37 chunks and storing in ChromaDB...
✅ Done — 37 vectors stored in ChromaDB
   Index saved to: ./chroma_lab_notebook/


### A6 — Step 6: Retrieve & Inspect Results

In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

QUESTION = "What are the main topics covered in this document?"

retrieved_docs = retriever.invoke(QUESTION)

print(f"❓ Question: {QUESTION}")
print(f"📋 Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"--- Chunk {i} (page {page_display}) ---")
    print(doc.page_content[:300])
    print()

❓ Question: What are the main topics covered in this document?
📋 Retrieved 4 chunks:

--- Chunk 1 (page 3) ---
CSD1130 – Game Implementation Techniques Lecture 11 | Binary Collision – Hot Spots  
 
 
© 2021, DigiPen Institute of Technology. All Rights Reserved. 3 
 
• Binary map collision can be used for other game types like Pacman. 
o Similarly, to how it is used in platform games, we can create a grid whe

--- Chunk 2 (page 4) ---
Lecture 11 | Binary Collision – Hot Spots  CSD1130 – Game Implementation Techniques 
 
4 © 2021, DigiPen Institute of Technology. All Rights Reserved. 
 
 
o "Map Data" is generally imported from a file which has been previously exported from 
an editor. Then "Collision Data" is constructed using "M

--- Chunk 3 (page 6) ---
▪ This done by ORing the collision flag with the correspondent collision side value.  
▪ At the end of the collision check, the flag variable can be used to determine 
which sides of the object instance have collided with solid areas. 

**🔑 This is the most important debugging step in RAG.** Before looking at the LLM's answer, always inspect what the retriever found. If these chunks don't contain the answer, the LLM won't be able to answer correctly — no prompt tuning will fix a retrieval problem.

### A7 — Step 7: Generate an Answer with Groq

In [14]:
from groq import Groq

SYSTEM_PROMPT = """You are a document-based AI assistant.
Answer ONLY using the retrieved context provided below.
For every fact, include the page number in parentheses, e.g. (page 2).
If the answer is not in the context, say: I could not find that in the provided document."""

def format_context(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        parts.append(f"[Source {i} | page {page_display}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, retriever, model="llama-3.1-8b-instant", temperature=0.2):
    """Full RAG loop: retrieve → build prompt → generate → return (answer, docs)."""
    docs    = retriever.invoke(question)
    context = format_context(docs)
    prompt  = f"""Question: {question}

Retrieved context:
{context}

Answer using ONLY the context above. Include page references."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content, docs

answer, retrieved = ask_rag(QUESTION, retriever)

print(f"❓ Question: {QUESTION}")
print()
print("📋 Retrieved chunks:")
for i, doc in enumerate(retrieved, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [{i}] page {page_display}: {doc.page_content[:80]}...")
print()
print("🤖 Answer:")
print(answer)

❓ Question: What are the main topics covered in this document?

📋 Retrieved chunks:
  [1] page 3: CSD1130 – Game Implementation Techniques Lecture 11 | Binary Collision – Hot Spo...
  [2] page 4: Lecture 11 | Binary Collision – Hot Spots  CSD1130 – Game Implementation Techniq...
  [3] page 6: ▪ This done by ORing the collision flag with the correspondent collision side va...
  [4] page 2: o It consists of dividing the level into a grid. 
o Each cell inside this grid c...

🤖 Answer:
The main topics covered in this document are:

1. Binary map collision and its application in different game types, such as Pacman (page 3).
2. Initialization of binary map collision, which relies on the game world being a grid (page 3).
3. Construction of "Collision Data" using "Map Data" (page 4).
4. Representation of a 5 by 5 grid in an array, where each cell with a value of 1 is a collision area (page 4).
5. Determining collision areas and sides of object instances using bitwise operations (page 6).
6. G

---
## 🅑 PART B — Experiments

Now you modify parameters and observe how they affect retrieval quality. **This is the core skill of RAG engineering.**

### B1 — ✏️ YOUR TURN: Chunk Size Experiment

Re-index your document with **three different chunk sizes** and compare how many chunks are created and how the retrieved text looks.  
Fill in the `build_index()` function below — it should work for all three sizes.

In [15]:
def build_index(pages, chunk_size, chunk_overlap, chroma_dir):
    """
    Build a ChromaDB index from pages with the given chunking parameters.
    Returns (vector_store, chunks)
    """
    # ✏️ YOUR TURN: create the splitter
    # Hint: RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=..., separators=[...])
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    # ✏️ YOUR TURN: split the pages into chunks
    chunks = splitter.split_documents(pages) 

    if Path(chroma_dir).exists():
        shutil.rmtree(chroma_dir)

    # ✏️ YOUR TURN: create the ChromaDB vector store
    # Hint: Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=chroma_dir, collection_name="exp")
    store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=chroma_dir, collection_name=f"chunks_{chunk_size}") 

    return store, chunks


# ── Run for three chunk sizes ─────────────────────────────────────────────────
test_question = "What are the main topics covered in this document?"
results = {}

for size in [300, 800, 1500]:
    overlap = int(size * 0.15)
    store, cks = build_index(pages, size, overlap, f"./chroma_exp_{size}")
    ret  = store.as_retriever(search_kwargs={"k": 3})
    docs = ret.invoke(test_question)
    results[size] = {"chunk_count": len(cks), "retrieved": docs}
    print(f"chunk_size={size:>4}: {len(cks):>4} chunks total | retrieved {len(docs)} docs")

print("\n✅ Experiment complete — run the next cell to compare retrieved text")

chunk_size= 300:   91 chunks total | retrieved 3 docs
chunk_size= 800:   37 chunks total | retrieved 3 docs
chunk_size=1500:   25 chunks total | retrieved 3 docs

✅ Experiment complete — run the next cell to compare retrieved text


In [16]:
# Inspect retrieved chunks for each size side-by-side
for size, data in results.items():
    print(f"\n{'='*60}")
    print(f"  chunk_size = {size}  ({data['chunk_count']} total chunks)")
    print(f"{'='*60}")
    for i, doc in enumerate(data["retrieved"], 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        print(f"  [{i}] page {page_display} | {len(doc.page_content)} chars")
        print(f"      {doc.page_content[:200]}")
        print()


  chunk_size = 300  (91 total chunks)
  [1] page 2 | 237 chars
      o Each level is divided into 2 main parts: Platforms and non-platforms (air). 
o World collision is “static”: Sprites can collide with the left side of a wall when moving 
right, or with the bottom si

  [2] page 4 | 280 chars
      an editor. Then "Collision Data" is constructed using "Map Data". 
o The example represents an array that divides the world into a 5 by 5 grid. 
o Since each cell, whose value is 1 is a collision area

  [3] page 7 | 283 chars
      a collision area. 
o It is considere d colliding with a collision area if at least 1 hot spot is inside a collision 
area, which means the collision flag has one or more bits set to 1. 
• If at least 


  chunk_size = 800  (37 total chunks)
  [1] page 3 | 715 chars
      CSD1130 – Game Implementation Techniques Lecture 11 | Binary Collision – Hot Spots  
 
 
© 2021, DigiPen Institute of Technology. All Rights Reserved. 3 
 
• Binary map collision can be used f

### 📝 Reflection B1

1. **How did chunk count change** as chunk_size increased from 300 → 800 → 1500?

   The chunk count decreased substantially as chunk size increased: `chunk_size=300` produced 91 chunks, `800` produced 37, and `1500` produced 25. Larger chunks combine more of each page's text into one retrieval unit, so fewer chunks are created.

2. **Which chunk size returned the most useful context** for your question? Why?

   `chunk_size=1500` returned the most useful context for the broad question about the document's main topics. Its results included the map/collision-data example, collision-flag logic, and a 1,444-character recapitulation section on page 13. The recapitulation supplied a more complete overview of the binary-collision workflow than the shorter fragments returned at sizes 300 and 800.

3. **With chunk_size=300**, did any retrieved chunk seem too short to be useful on its own?

   Yes. The three retrieved chunks were only 237–283 characters and each captured a small fragment, such as level structure, construction of collision data, or the hot-spot collision rule. Each fragment was relevant, but none gave a complete overview on its own; neighbouring chunks would be needed to understand the full process.

4. **With chunk_size=1500**, did the retrieved chunk contain irrelevant sentences alongside the relevant ones?

   Not significantly. All three results were related to binary collision: collision/map data, collision flags, and the recapitulation. The first two chunks were only 298 and 363 characters because the splitter preserved page and natural-text boundaries, while the page-13 chunk used most of the 1,500-character allowance. The larger size improved completeness here without introducing obvious off-topic material.

### B2 — ✏️ YOUR TURN: The k Parameter Experiment

Using a `chunk_size=800` index, compare what happens when you retrieve **k=1**, **k=4**, and **k=10** chunks.

In [17]:
import uuid

# Use a fresh directory so rerunning this cell does not try to delete an open ChromaDB index.
chroma_k_dir = f"./chroma_k_exp_{uuid.uuid4().hex[:8]}"
store_800, _ = build_index(pages, 800, 120, chroma_k_dir)

# ✏️ YOUR TURN: pick a question relevant to YOUR document
question = "What are the main components and steps of a RAG pipeline?"

print(f"Question: {question}\n")

for k in [1, 4, 10]:
    # ✏️ YOUR TURN: create a retriever with this k value and retrieve docs
    # Hint: store_800.as_retriever(search_kwargs={"k": k})
    retriever_k = store_800.as_retriever(search_kwargs={"k": k})
    docs_k      = retriever_k.invoke(question)

    total_chars = sum(len(d.page_content) for d in docs_k)
    pages_found = [d.metadata.get('page', '?') for d in docs_k]
    print(f"k={k:2d} → {len(docs_k)} chunks | ~{total_chars} chars of context | pages: {pages_found}")

Question: What are the main components and steps of a RAG pipeline?

k= 1 → 1 chunks | ~325 chars of context | pages: [5]
k= 4 → 4 chunks | ~1744 chars of context | pages: [5, 2, 7, 3]
k=10 → 10 chunks | ~5802 chars of context | pages: [5, 2, 7, 3, 3, 6, 3, 12, 7, 1]


In [18]:
# Compare the ANSWERS generated with k=1 vs k=4 vs k=10
for k in [1, 4, 10]:
    ret_k    = store_800.as_retriever(search_kwargs={"k": k})
    ans_k, _ = ask_rag(question, ret_k)
    print(f"\n{'─'*60}")
    print(f"  k = {k}")
    print(f"{'─'*60}")
    print(ans_k)


────────────────────────────────────────────────────────────
  k = 1
────────────────────────────────────────────────────────────
I could not find information about the main components and steps of a RAG pipeline in the provided document.

────────────────────────────────────────────────────────────
  k = 4
────────────────────────────────────────────────────────────
Based on the provided context, I could not find a detailed explanation of the main components and steps of a RAG (Ray Casting Algorithm) pipeline. However, I can infer some general steps from the context:

1. **Map Data and Collision Data**: The context mentions "Map Data" and "Collision Data" arrays, which are used to store the map and collision information, respectively (Source 2, page 3).
2. **Collision Check**: The context describes a collision check between an object instance and the binary collision map, which involves checking for collision between hotspots (points) and the map (Source 4, page 4).
3. **Transformati

### 📝 Reflection B2

1. **k=1 vs k=4:** Did increasing k improve the answer? In what way?

   No. The question was out of scope because the new PDF covers binary collision rather than Retrieval-Augmented Generation. With `k=1`, the model received one 325-character chunk and correctly refused. With `k=4`, it received 1,744 characters of unrelated collision material and produced a long inferred answer, incorrectly interpreting RAG as a ray-casting algorithm and adding prior knowledge. Increasing `k` therefore reduced grounding quality rather than improving the answer.

2. **k=10:** Better or worse than k=4? Any sign of context dilution?

   `k=10` was somewhat better than `k=4` because it explicitly stated that a RAG pipeline was not described and then summarized the actual binary-collision process. However, the 5,802 characters of context still did not contain an answer to the RAG question, and the response supplied an alternative answer the user did not request. This is evidence that a larger `k` cannot repair an out-of-scope query and may dilute the context with more irrelevant material.

3. **If you had to pick one k value**, which would you choose and why?

   For this out-of-scope question, I would choose `k=1` because it produced the only clean refusal and avoided turning unrelated collision text into a fabricated RAG explanation. This result should not be used to select a general production value of `k`; that requires relevant, in-scope questions. The experiment instead shows that relevance checks and refusal behavior matter more than retrieving additional chunks when the collection does not contain the answer.

### B3 — ✏️ YOUR TURN: Temperature Experiment

In [19]:
retriever_b3 = store_800.as_retriever(search_kwargs={"k": 4})
question_b3  = question  # reuse your question from B2

for temp in [0.0, 0.5, 1.0]:
    # ✏️ YOUR TURN: call ask_rag with the current temperature
    # Hint: ask_rag(question, retriever, temperature=temp)
    answer_t, _ = ask_rag(question_b3, retriever_b3, temperature=temp)

    print(f"\n{'─'*60}")
    print(f"  temperature = {temp}")
    print(f"{'─'*60}")
    print(answer_t)


────────────────────────────────────────────────────────────
  temperature = 0.0
────────────────────────────────────────────────────────────
Based on the provided context, I could not find a detailed explanation of the main components and steps of a RAG (Ray Casting Algorithm) pipeline. However, I can infer some general steps from the context.

The context mentions the following steps related to collision detection and rendering:

1. **Collision Detection**: Checking for collision between an object instance and the binary collision map (Source 4, page 4). This involves testing for collision between hotspots (points) on the object instance and the binary collision map.
2. **Collision Data**: Storing collision data in a separate array (Source 2, page 3).
3. **Transformation**: Translating and scaling the binary map to the correct position in the world (Source 3, page 8).
4. **Rendering**: Rendering the transformed binary map (Source 3, page 8).

However, these steps do not explicitly f

### 📝 Reflection B3

1. **temperature=0.0 vs 1.0:** How did the answer style differ? Was it more or less consistent on re-run?

   At temperature 0.0, the answer was the longest and most speculative: it incorrectly expanded RAG as a ray-casting algorithm and added scene graphs and collision response from outside the retrieved document. At temperature 1.0, the answer stayed closer to the collision-map evidence and clearly noted that RAG was not explicitly mentioned, although it still answered with an inferred collision pipeline. Only one saved run is available, so consistency across repeated runs cannot be concluded from these outputs.

2. **For a factual document Q&A system**, which temperature range would you recommend and why?

   I would still start with a low range such as 0.0–0.2 for factual document Q&A because it generally favors repeatability, but this experiment shows that low temperature does not guarantee faithfulness: the temperature-0.0 answer hallucinated the most. Retrieval relevance, explicit refusal rules, and preventing the model from using prior knowledge are more important than temperature when the requested information is absent.

### B4 — 🔬 EXPERIMENT: Ask an Out-of-Scope Question

In [20]:
retriever_b4   = store_800.as_retriever(search_kwargs={"k": 4})
out_of_scope_q = "What is the current price of Bitcoin in Singapore dollars?"

answer_oos, docs_oos = ask_rag(out_of_scope_q, retriever_b4)

print(f"❓ Question: {out_of_scope_q}")
print()
print("📋 What the retriever found (likely unrelated):")
for doc in docs_oos:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  page {page_display}: {doc.page_content[:100]}...")
print()
print("🤖 Answer:")
print(answer_oos)

❓ Question: What is the current price of Bitcoin in Singapore dollars?

📋 What the retriever found (likely unrelated):
  page 16: Lecture 11 | Binary Collision – Hot Spots  CSD1130 – Game Implementation Techniques 
 
16 © 2021, Di...
  page 6: ▪ Each game object instance will have a collision flag, where each  bit represents 
one side. 
• The...
  page 6: Lecture 11 | Binary Collision – Hot Spots  CSD1130 – Game Implementation Techniques 
 
6 © 2021, Dig...
  page 15: CSD1130 – Game Implementation Techniques Lecture 11 | Binary Collision – Hot Spots  
 
 
© 2021, Dig...

🤖 Answer:
I could not find the current price of Bitcoin in Singapore dollars in the provided document.


### 📝 Reflection B4

1. Did the model correctly refuse, or did it hallucinate using irrelevant context?

   The model correctly refused and did not hallucinate. The retriever returned unrelated binary-collision chunks from pages 6, 15, and 16, but the answer stated that the current Bitcoin price in Singapore dollars could not be found in the document. It did not invent a price or misuse the collision content.

2. If it hallucinated: what in `SYSTEM_PROMPT` could you change to make refusal more reliable?

   The model did not hallucinate, so no corrective prompt change was required for this run. If refusal needed to be made even more reliable, I would explicitly prohibit using prior knowledge or guessing and require the exact refusal response whenever the retrieved context does not directly support the answer.

3. Modify `SYSTEM_PROMPT` in cell A7 to make refusal more explicit, re-run, and report what changed:

   The existing system prompt already instructed the model to answer only from the retrieved context and to refuse when the answer was absent. The re-run produced the correct refusal: "I could not find the current price of Bitcoin in Singapore dollars in the provided document." Therefore, the prompt successfully prevented an unsupported answer; no observable correction was needed for this test.

---
## 🅒 PART C — Open Exploration

### C1 — ✏️ YOUR TURN: Test Your Own PDF

In [21]:
# Load and index a different PDF, then ask three questions.
import uuid

MY_PDF = "YOUR_PDF_HERE.pdf"

# Use a fresh directory on every run to avoid Windows file-lock errors from ChromaDB.
c1_chroma_dir = f"./chroma_c1_{uuid.uuid4().hex[:8]}"
my_pages = PyPDFLoader(MY_PDF).load()
my_store, my_chunks = build_index(
    my_pages,
    chunk_size=800,
    chunk_overlap=120,
    chroma_dir=c1_chroma_dir,
)
my_retriever = my_store.as_retriever(search_kwargs={"k": 4})

questions_c1 = [
    "What is the main purpose and topic of this document?",
    "What are the most important findings or recommendations in this document?",
    "What conclusions does the document make?",
]

print(f"Loaded {len(my_pages)} pages and created {len(my_chunks)} chunks.")

for number, question_c1 in enumerate(questions_c1, start=1):
    answer_c1, docs_c1 = ask_rag(question_c1, my_retriever)
    pages_c1 = [
        doc.metadata.get("page") + 1
        if isinstance(doc.metadata.get("page"), int)
        else "?"
        for doc in docs_c1
    ]

    print(f"\n{'=' * 70}")
    print(f"Q{number}: {question_c1}")
    print(f"Retrieved pages: {pages_c1}")
    for i, doc in enumerate(docs_c1, start=1):
        preview = doc.page_content[:180].replace("\n", " ")
        print(f"  Chunk {i}: {preview}...")
    print(f"\nAnswer:\n{answer_c1}")

Loaded 446 pages and created 1300 chunks.

Q1: What is the main purpose and topic of this document?
Retrieved pages: [28, 116, 127, 118]
  Chunk 1: Part 1: The basics...
  Chunk 2: 1. The Purpose and character of the use: ifsomeonecandemonstratethattheiruseadvancesknowledgeor theprogressofartsthroughtheadditionofsomethingnew,it’sprobablyfairuse. Thisusuallyis...
  Chunk 3: “TheResistance”. Strongofthousandsofmembersandthecollaborationof the Capacitance,theresistancelaunchedanattackagainst theevilreactanceempire,buttheempirestrokebackwithacarpetsurcha...
  Chunk 4: ThismakestheWaterfalllifecyclemodel extremely rigid,everythingneedstobecarefullyanalyzedanddocumented (sometimespeopledefinethismodel“document-driven”)andthecodingisdoneonlyinitsfi...

Answer:
The main purpose of this document appears to be a guide for game development, specifically project management basics and tips. 

The topic of this document is likely game development, project management, and possibly copyright laws in r

### 📝 Reflection C1

| Question | Retrieval correct? | Answer quality (1–5) | Notes |
|---|---|---|---|
| Q1: What is the main purpose and topic of this document? | Partly | 3 | Retrieval found a section heading, project-management material, copyright information, and game-related content. The answer reasonably identified game development as the overall topic, but the retrieved chunks were scattered and did not provide a clear statement of purpose. |
| Q2: What are the most important findings or recommendations in this document? | No | 2 | Retrieval mostly returned design-pattern summary tables, a broad heading, and one Waterfall-model passage. The model appropriately warned that it could not find a clear findings or recommendations section, but the resulting answer did not fully answer the question. |
| Q3: What conclusions does the document make? | No | 2 | The retrieved chunks came from summary tables and unrelated discussions of Waterfall and A/B testing rather than a conclusion section. The model correctly refused instead of inventing a conclusion, but retrieval did not supply the evidence needed to answer. |

**Best chunk_size and k for your document?** The best settings cannot be determined from C1 because only `chunk_size=800` and `k=4` were tested. This configuration was an acceptable baseline, but it performed poorly on broad summary and conclusion questions for a 446-page document. I would compare additional chunk sizes and higher `k` values, and use more specific questions or hierarchical retrieval before selecting the final settings.

### C2 — 🔬 EXPERIMENT: Multi-Query Retrieval

In [22]:
import re

# Generate alternative phrasings without relying on a version-specific LangChain retriever.
test_q = question  # reuse the question from B2
rewrite_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
rewrite_response = rewrite_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{
        "role": "user",
        "content": (
            "Rewrite the following question in three different ways for semantic "
            "document retrieval. Return exactly three rewrites, one per line, "
            f"with no explanation. Question: {test_q}"
        ),
    }],
    temperature=0.2,
)

raw_rewrites = rewrite_response.choices[0].message.content.splitlines()
rewrites = [
    re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
    for line in raw_rewrites
    if line.strip()
][:3]
query_variants = [test_q, *rewrites]

base_retriever = store_800.as_retriever(search_kwargs={"k": 4})
docs_standard = base_retriever.invoke(test_q)

# Retrieve for every phrasing and deduplicate chunks by their text.
docs_multi = []
seen_texts = set()
for query_variant in query_variants:
    for doc in base_retriever.invoke(query_variant):
        if doc.page_content not in seen_texts:
            seen_texts.add(doc.page_content)
            docs_multi.append(doc)

print("Query variants:")
for i, query_variant in enumerate(query_variants, start=1):
    print(f"  {i}. {query_variant}")
print()

print(f"Standard retrieval : {len(docs_standard)} chunks")
print(f"Multi-query        : {len(docs_multi)} chunks (after dedup)")

standard_texts = {d.page_content for d in docs_standard}
extra_chunks   = [d for d in docs_multi if d.page_content not in standard_texts]
print(f"\nExtra chunks found ONLY by multi-query: {len(extra_chunks)}")
for doc in extra_chunks:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [page {page_display}] {doc.page_content[:150]}")

Query variants:
  1. What are the main components and steps of a RAG pipeline?
  2. What are the key components and stages involved in a RAG pipeline implementation?
  3. What are the primary elements and procedures of a RAG pipeline architecture?
  4. What constitutes the essential building blocks and workflow of a RAG pipeline system?

Standard retrieval : 4 chunks
Multi-query        : 9 chunks (after dedup)

Extra chunks found ONLY by multi-query: 5
  [page 3] CSD1130 – Game Implementation Techniques Lecture 11 | Binary Collision – Hot Spots  
 
 
© 2021, DigiPen Institute of Technology. All Rights Reserved.
  [page 4] the top left depending on how the map was exported from the map editor. More on 
that later). 
o This array can be static for games like Pacman where 
  [page 4] Lecture 11 | Binary Collision – Hot Spots  CSD1130 – Game Implementation Techniques 
 
4 © 2021, DigiPen Institute of Technology. All Rights Reserved.
  [page 2] o Usually, a bounding circle or rectangle. 
o 

### C3 — 🔬 EXPERIMENT: Retrieval with Similarity Scores

In [23]:
results_with_scores = vector_store.similarity_search_with_score(question, k=4)

print(f"Question: {question}\n")
print(f"{'Similarity':>10}  {'Page':>5}  Content preview")
print("-" * 72)
for doc, score in results_with_scores:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    similarity   = 1 / (1 + score)  # convert L2 distance to rough similarity
    print(f"{similarity:>10.4f}  {str(page_display):>5}  {doc.page_content[:60]}...")

Question: What are the main components and steps of a RAG pipeline?

Similarity   Page  Content preview
------------------------------------------------------------------------
    0.3628      6  ▪ This done by ORing the collision flag with the corresponde...
    0.3600      3  • The other array called “Collision Data” will just hold the...
    0.3582      8  coordinates system of the binary map before actually renderi...
    0.3571      4  o Generally, they are made from a group of triangles. 
• So ...


### 📝 Reflection C — Overall

1. Is there a clear similarity score gap between the most and least relevant chunks?

   No. The four similarity values were 0.3628, 0.3600, 0.3582, and 0.3571, so the gap between the strongest and weakest result was only 0.0057. All four results were weak, nearly tied matches from the binary-collision document, which contains no RAG pipeline. C2 reinforces this result: multi-query retrieval increased the unique result count from 4 to 9, but all five additional chunks were still unrelated collision material. Rewriting a query can improve coverage, but it cannot create relevant evidence that is absent from the collection.

2. **Two most important RAG parameters to tune** for a new document collection?

   The two most important parameters to tune first are `chunk_size` and retrieval `k`. Chunk size determines whether each unit contains a complete idea or only a fragment; in B1, size 1500 retrieved the most useful recapitulation, while size 300 returned partial snippets. The `k` value controls evidence coverage and noise; in B2, increasing `k` for an out-of-scope question added unrelated context and encouraged hallucination. Parameter tuning must therefore use representative in-scope and out-of-scope questions, not retrieval count alone.

3. **Which mini-project did you choose, and why?** What do you expect to be hardest?

   I chose a document Q&A assistant for a game-development reference book because the 446-page document contains material on game design, project management, software-development processes, copyright, and design patterns. The hardest part will be retrieving coherent evidence from 1,300 chunks. C1 showed that broad summary questions often matched headings, index entries, and isolated passages; C2 also showed that multi-query retrieval can increase the amount of irrelevant material when the query is outside the collection's scope. More specific questions, hierarchical retrieval, metadata filtering, and relevance thresholds would likely be needed.

---
## 🌐 PART D — Web-Augmented RAG

### What is Web-Augmented RAG?

So far, every answer has been grounded in a **local PDF** that you indexed yourself. But what if the user asks something that is not in any of your documents — for example, a current event, a recent product release, or a live regulation?

**Web-Augmented RAG** solves this by replacing (or combining) the vector-store retriever with a **live web search**. Instead of fetching chunks from ChromaDB, the system searches the web in real time, retrieves the top results, and uses those as the LLM's context.

```
Standard RAG:          Question → ChromaDB → chunks → LLM → answer
Web-Augmented RAG:     Question → Web search → live results → LLM → answer
Combined (best):       Question → ChromaDB + Web search → merge → LLM → answer
```

### Tool we use: Tavily Search

**Tavily** is a search API designed specifically for LLM applications. It returns clean, structured, LLM-friendly text from web results — not raw HTML. LangChain has a native `TavilySearchResults` integration.

**Free tier:** 1,000 searches/month — more than enough for this lab.  
**Sign up:** https://tavily.com → get your API key in 30 seconds.

### D0 — Install & Set Your Tavily Key

In [24]:
# Install the Tavily integration
%pip install tavily-python langchain-community
print("✅ Tavily installed")

Note: you may need to restart the kernel to use updated packages.
✅ Tavily installed


In [25]:
# Set your Tavily API key
# Get it from: https://app.tavily.com/home  (free, takes ~30 seconds)
#
# Option A — add to your .env file:    TAVILY_API_KEY=tvly-...
# Option B — set it here for this session only (do not commit this notebook with the key visible):
#
from dotenv import load_dotenv, find_dotenv
env_path = find_dotenv()
load_dotenv(env_path, override=True)  # reads TAVILY_API_KEY from .env if present
# os.environ["TAVILY_API_KEY"] = "tvly-your-key-here"

tavily_key = os.getenv("TAVILY_API_KEY") #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
if tavily_key:
    print(f"✅ TAVILY_API_KEY found (starts with: {tavily_key[:8]}...)")
else:
    print("❌ TAVILY_API_KEY not set.")
    print("   Get a free key at https://app.tavily.com/home")
    print("   Then add TAVILY_API_KEY=tvly-... to your .env file and re-run this cell.")

✅ TAVILY_API_KEY found (starts with: tvly-dev...)


### D1 — Guided: Web Search as a Retriever

Here we use `TavilySearchResults` as a drop-in retriever. The API returns a list of result objects, each with `content` (the page text) and `url`. We convert these into `Document` objects so the rest of our pipeline stays identical — the same `format_context()` and `ask_rag()` functions work unchanged.

In [26]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document

# Create the web search tool
# max_results: how many web pages to retrieve per query (3–5 is a good range)
web_search = TavilySearchResults(
    max_results=4,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

def web_search_to_docs(query: str) -> list:
    """
    Run a web search and convert results to Document objects
    so they work with our existing format_context() function.
    Each Document gets:
      - page_content: the retrieved text snippet
      - metadata["source"]: the URL
      - metadata["page"]: "web" (instead of a page number)
    """
    results = web_search.invoke(query)
    docs = []
    for r in results:
        docs.append(Document(
            page_content=r["content"],
            metadata={"source": r["url"], "page": "web"},
        ))
    return docs

# Test with a query that definitely requires current/live information
web_query = "Latest developments in AI regulation in Singapore 2025"
web_docs  = web_search_to_docs(web_query)

print(f"🌐 Web search: '{web_query}'")
print(f"   Retrieved {len(web_docs)} results\n")
for i, doc in enumerate(web_docs, 1):
    print(f"  [{i}] Source: {doc.metadata['source']}")
    print(f"       {doc.page_content[:200]}")
    print()

C:\Users\edgar\AppData\Local\Temp\ipykernel_23440\542086635.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(


🌐 Web search: 'Latest developments in AI regulation in Singapore 2025'
   Retrieved 4 results

  [1] Source: https://regulations.ai/regulations/RAI-SG-NA-SUMMARY-2026
       Singapore's AI regulatory landscape is dynamic and continuously evolving, with several key future developments anticipated. The Online Safety (Relief and Accountability) Bill (OSRA Bill) is currently 

  [2] Source: https://digital.nemko.com/regulations/singapore-ai-regulation
       ## 

## High-Impact and Generative AI Oversight

Singapore’s regulatory focus in 2025 extends to the governance of high-impact and generative AI systems, emphasizing robust safety testing, accountabil

  [3] Source: https://www.linkedin.com/posts/nicholasker_ai-regulations-in-2025-is-singapore-getting-activity-7310812823066984448-w7Hz
       AI Regulations in 2025: Is Singapore getting it right? AI is transforming industries at lightning speed, and with that comes a wave of evolving regulations. Hyperight’s latest article lays out six 

### D2 — Guided: Generate an Answer from Web Results

We can now use the same `format_context()` and the same Groq call — just swapping the source of the context from ChromaDB chunks to live web results.

In [27]:
WEB_SYSTEM_PROMPT = """You are a research assistant answering questions using live web search results.
Answer using ONLY the retrieved web content provided below.
For every fact, cite the source URL in parentheses.
If the information is not in the provided results, say: I could not find that in the retrieved web content.
Do NOT use your own training knowledge."""

def ask_web_rag(question: str, max_results: int = 4) -> str:
    """Full Web-RAG loop: web search → format context → Groq → answer."""
    # Step 1: retrieve from the web
    docs    = web_search_to_docs(question)
    context = format_context(docs)  # reusing the same function from Part A

    # Step 2: build prompt and generate
    prompt = f"""Question: {question}

Retrieved web content:
{context}

Answer using ONLY the retrieved content above. Cite the source URL for every claim."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": WEB_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Ask something that requires live, current information
live_question = "What are the latest AI safety regulations announced in 2025?"

print(f"❓ Question: {live_question}")
print()
answer_web = ask_web_rag(live_question)
print("🤖 Answer (grounded in live web results):")
print(answer_web)

❓ Question: What are the latest AI safety regulations announced in 2025?

🤖 Answer (grounded in live web results):
According to the retrieved web content, the latest AI safety regulations announced in 2025 include:

1. China's mandatory labeling rule for AI-generated content, which was issued by the Cyberspace Administration of China (CAC) in March 2025 and takes effect on September 1, 2025. (Source 1 | page web)
2. The European Union's AI Act, which takes a risk-based approach and prohibits certain AI practices outright, such as real-time biometric identification in public spaces for law enforcement. (Source 1 | page web)
3. The US federal government's Executive Order 14179, which replaced Executive Order 14110 and shifts toward deregulation and prioritizes AI innovation and US competitiveness. (Source 3 | page web)
4. The New York Responsible AI Safety and Education (RAISE) Act, which was enacted on December 19, 2025, and imposes transparency obligations on large developers of advanc

### D3 — ✏️ YOUR TURN: Compare Local RAG vs Web RAG on the Same Question

Now ask the **same question** to both your local ChromaDB retriever and the web search retriever.  
Choose a question where your local PDF might have partial information but the web would have more current details —  
for example: a topic your PDF covers, but the web has newer developments on.

**Goal:** understand when to use each approach, and when to combine them.

In [28]:
# ✏️ YOUR TURN: pick a question that makes sense for both your local PDF AND the web
# Good examples:
#   - A topic your PDF covers historically, but the web has 2025 updates on
#   - A regulation your PDF mentions, but whose current status you want to verify
#   - A technology your PDF explains, but recent benchmarks exist online

comparison_question = "What are the latest advances in RAG beyond naive RAG?"

# ── Local RAG (your PDF) ──────────────────────────────────────────────────────
# ✏️ YOUR TURN: use ask_rag() with store_800's retriever
local_retriever = store_800.as_retriever(search_kwargs={"k": 4})
local_answer, local_docs = ask_rag(comparison_question, local_retriever)

print("=" * 60)
print("📄 LOCAL RAG (from your PDF)")
print("=" * 60)
local_pages = [
    d.metadata.get("page") + 1
    if isinstance(d.metadata.get("page"), int)
    else "?"
    for d in local_docs
]
print("Sources used:", [f"page {page}" for page in local_pages])
print(local_answer)

print()
print("=" * 60)
print("🌐 WEB RAG (live web search)")
print("=" * 60)

# ✏️ YOUR TURN: use ask_web_rag() with the same question
web_answer = ask_web_rag(comparison_question)
print(web_answer)

📄 LOCAL RAG (from your PDF)
Sources used: ['page 3', 'page 19', 'page 16', 'page 13']
I could not find any information on the latest advances in RAG (Recurrent Attention Generator) beyond naive RAG in the provided context.

🌐 WEB RAG (live web search)
The latest advances in RAG beyond naive RAG include:

1. Advanced RAG with optimized retrieval strategies (Source 1: RAG Survey).
2. Modular RAG with flexible, task-specific architectures (Source 1: RAG Survey).
3. Graph Foundation Model for RAG (GFM-RAG) using Graph Neural Networks (GNNs) to refine connections between queries on niche datasets (Source 3: GFM-RAG).
4. Contextual Retrieval, which reduces retrieval failures by 67% (Source 4: FAQs about advanced RAG techniques).
5. RAPTOR, which improves absolute accuracy on QuALITY by +20% (Source 4: FAQs about advanced RAG techniques).
6. Self-RAG, which beats standard RAG on open-domain QA (Source 4: FAQs about advanced RAG techniques).
7. GraphRAG, which enables multi-hop sensemaking (So

### D4 — 🔬 EXPERIMENT: Combining Both (Hybrid Retrieval)

The most powerful approach is to merge results from both your local index AND the web,  
then let the LLM synthesise across all sources. This is how production systems like Perplexity work.

This cell is **fully written** — just run it and observe how the answer changes when the LLM has both local and web context.

In [29]:
HYBRID_SYSTEM_PROMPT = """You are a research assistant with access to both a local document library and live web search results.
You will receive context from two sources: LOCAL DOCUMENT and WEB SEARCH.
Synthesise information from both sources to give the most complete and accurate answer.
Clearly label which source each piece of information comes from:
  - Local: cite the page number
  - Web: cite the URL
If the sources contradict each other, note the discrepancy and explain which seems more current."""

def ask_hybrid_rag(question: str, local_retriever, k_web: int = 3) -> str:
    """Retrieves from both local ChromaDB and web, then generates a combined answer."""
    # Get local chunks
    local_docs = local_retriever.invoke(question)
    # Get web results
    web_docs   = web_search_to_docs(question)

    # Format each source separately so the LLM can attribute correctly
    local_context = format_context(local_docs)
    web_context   = format_context(web_docs)

    prompt = f"""Question: {question}

=== LOCAL DOCUMENT CONTEXT ===
{local_context}

=== LIVE WEB SEARCH CONTEXT ===
{web_context}

Synthesise both sources to answer the question. Label each fact with its source (page number or URL)."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": HYBRID_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Run with the same question from D3
hybrid_retriever = store_800.as_retriever(search_kwargs={"k": 3})
hybrid_answer    = ask_hybrid_rag(comparison_question, hybrid_retriever)

print("=" * 60)
print("🔀 HYBRID RAG (local PDF + live web)")
print("=" * 60)
print(hybrid_answer)

🔀 HYBRID RAG (local PDF + live web)
Based on the provided sources, the latest advances in RAG beyond naive RAG include:

1. **Advanced RAG**: Incorporates optimization strategies for pre-retrieval and post-retrieval, allowing for more efficient and relevant retrieval (Source 2, page web).
2. **Modular RAG**: Introduces a more flexible approach, including fine-tuning the retriever for enhanced adaptability (Source 2, page web).
3. **Contextual Retrieval**: Dynamically decides retrieval steps, incorporating multimodal data, and optimizes resource use (Source 3, page web).
4. **RAPTOR**: Improves absolute accuracy on QuALITY by +20% (Source 4, page web).
5. **Self-RAG**: Beats standard RAG on open-domain QA, using a self-reflection mechanism (Source 4, page web).
6. **GraphRAG**: Enables multi-hop sensemaking using a graph foundation model (Source 3, page web).
7. **Hybrid Retrieval**: Combines multiple retrieval strategies for improved accuracy and efficiency (Source 3, page web).
8. **A

### 📝 Reflection D

1. **Local vs Web:** For your chosen question, which source gave a more complete answer?

   The web source gave the only complete answer. The local binary-collision PDF contained no information about advances in Retrieval-Augmented Generation, so local RAG correctly refused after retrieving unrelated pages 3, 13, 16, and 19. Web RAG returned current material on Advanced and Modular RAG, GFM-RAG, Contextual Retrieval, RAPTOR, Self-RAG, GraphRAG, Hybrid Retrieval, and Adaptive RAG. This demonstrates that source selection must match the question: local RAG cannot answer a topic absent from its indexed collection.

2. **Hybrid answer:** Did combining both sources produce a noticeably better answer than either alone? What did each source contribute?

   The hybrid answer was not materially better than web RAG for factual coverage because the local PDF contributed no relevant RAG information. Its useful contribution was diagnostic: it explicitly identified the local map, collision-data, and sprite-position passages as unrelated and relied on live web content for the answer. The hybrid output therefore demonstrated graceful source separation, but it also added unnecessary local context and processing. For this question, web-only RAG was the cleaner choice.

3. **When would you choose each approach in a real product?**

   | Scenario | Best approach | Reason |
   |---|---|---|
   | Internal HR policy bot | Local RAG | Policy is private, not on the web |
   | Current news summariser | Web RAG | News changes rapidly, so live web retrieval is needed for current information. |
   | Product manual Q&A with recent updates | Hybrid RAG | The local manual supplies authoritative instructions, while web retrieval can add recent fixes, releases, and updates. |
   | General research assistant | Hybrid RAG | Combining curated local sources with current web material gives broader coverage and allows comparison of established and recent information. |

4. **What is one risk** of using web-augmented RAG that doesn't exist with local RAG?

   One risk is unreliable web content. Unlike a controlled local collection, live search can retrieve inaccurate, outdated, biased, or malicious pages, including content designed to manipulate an AI system. Web-derived claims therefore require source-quality checks, citation verification, and protection against prompt injection before they are trusted.

---
## 🧪 PART E — RAG Evaluation with RAGAS  *(Optional — Stretch Goal)*

> **This section is optional.** Complete Parts A–D first. If you have time remaining, come back here.

### Why evaluate?

So far you have been evaluating RAG the hard way: reading retrieved chunks and answers manually and forming a gut feeling about quality. That works for a few questions — but in a real product with thousands of queries, you need **automated, reproducible metrics**.

### What is RAGAS?

**RAGAS** (Retrieval Augmented Generation Assessment) is an open-source Python library that automatically scores your RAG pipeline on four key metrics — using an LLM as a judge. It requires no human-labelled ground truth for the basic metrics.

| Metric | What it asks | Score range |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? (no hallucination) | 0–1 (higher = better) |
| **Answer Relevancy** | Does the answer actually address the user's question? | 0–1 (higher = better) |
| **Context Recall** | Did retrieval find all the evidence needed to answer? | 0–1 (higher = better) |
| **Context Precision** | Are the retrieved chunks relevant? Low = noisy retrieval | 0–1 (higher = better) |

We will run **Faithfulness** and **Answer Relevancy** — the two that don't require ground-truth reference answers, making them the easiest to use.

### E0 — Install RAGAS

In [30]:
%pip install ragas datasets langchain-groq
print("✅ RAGAS installed")

Note: you may need to restart the kernel to use updated packages.
✅ RAGAS installed


### E1 — Guided: Understand the RAGAS Input Format

RAGAS expects a dataset where each row represents one Q&A exchange and contains:

| Field | Type | Description |
|---|---|---|
| `user_input` | `str` | The question that was asked |
| `response` | `str` | The LLM's answer |
| `retrieved_contexts` | `list[str]` | The raw text of each retrieved chunk (as plain strings) |
| `reference` | `str` | *(Optional)* Ground-truth answer — needed for Context Recall but not Faithfulness/Relevancy |

The cell below shows how to build this dataset from your existing `ask_rag()` function.

In [31]:
# This cell is fully written — run it to see what RAGAS data looks like

# We'll run 3 test questions through our local RAG pipeline
# and collect (question, answer, retrieved_chunks) for each

eval_retriever = store_800.as_retriever(search_kwargs={"k": 4})

# Define a few representative test questions for your document
# 📌 Change these to questions that make sense for YOUR PDF
test_questions = [
    "What are the main topics covered in this document?",
    "What is the current price of Bitcoin?",         # out-of-scope — should score low on faithfulness
    "What does the document say about deadlines?",   # replace with something relevant to your PDF
]

# Collect results
eval_rows = []
for q in test_questions:
    answer, docs = ask_rag(q, eval_retriever)
    eval_rows.append({
        "user_input"          : q,
        "response"            : answer,
        "retrieved_contexts"  : [doc.page_content for doc in docs],  # plain strings, not Document objects
    })
    print(f"✅ Collected: {q[:60]}...")

print(f"\n📊 {len(eval_rows)} rows ready for evaluation")
print()
print("Sample row (first question):")
print(f"  user_input         : {eval_rows[0]['user_input']}")
print(f"  response (first 100): {eval_rows[0]['response'][:100]}...")
print(f"  retrieved_contexts : {len(eval_rows[0]['retrieved_contexts'])} chunks")

✅ Collected: What are the main topics covered in this document?...
✅ Collected: What is the current price of Bitcoin?...
✅ Collected: What does the document say about deadlines?...

📊 3 rows ready for evaluation

Sample row (first question):
  user_input         : What are the main topics covered in this document?
  response (first 100): The main topics covered in this document are:

1. Binary map collision and its application in game t...
  retrieved_contexts : 4 chunks


### E2 — ✏️ YOUR TURN: Run the RAGAS Evaluation

Now you will build the RAGAS dataset and run evaluation. Fill in the three gaps marked `# ✏️ YOUR TURN`.

**What to expect:**
- The evaluation takes ~30–60 seconds — RAGAS uses an LLM internally to score each answer
- By default RAGAS uses OpenAI. We configure it to use Groq instead (free)
- Scores will be between 0 and 1 — higher is better
- The out-of-scope Bitcoin question should score **low** on faithfulness (the model likely hallucinated)

In [32]:
import sys
import types

# RAGAS 0.4.3 imports a VertexAI module removed from langchain-community 0.4.x.
# E2 uses Groq, not VertexAI, so provide the missing import symbol without
# downgrading LangChain or modifying installed package files.
vertexai_compat = "langchain_community.chat_models.vertexai"
if vertexai_compat not in sys.modules:
    vertexai_module = types.ModuleType(vertexai_compat)
    vertexai_module.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules[vertexai_compat] = vertexai_module

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# ── Configure RAGAS to use Groq + local embeddings (both free) ─────────────────
ragas_llm = LangchainLLMWrapper(ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
))
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

# ── ✏️ YOUR TURN 1: Build the Hugging Face Dataset from eval_rows ──────────────
# Hint: Dataset.from_list(eval_rows)
eval_dataset = Dataset.from_list(eval_rows)

print(f"Dataset columns : {eval_dataset.column_names}")
print(f"Dataset rows    : {len(eval_dataset)}")
print()

# ── ✏️ YOUR TURN 2: Run the evaluation ─────────────────────────────────────────
# Hint: evaluate(
#     dataset=eval_dataset,
#     metrics=[faithfulness, answer_relevancy],
#     llm=ragas_llm,
#     embeddings=ragas_embeddings,
# )
print("Running RAGAS evaluation (takes ~30–60 seconds)...")
results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n✅ Evaluation complete!")
print()

# ── ✏️ YOUR TURN 3: Print the results as a readable table ──────────────────────
# Hint: results.to_pandas() gives you a DataFrame
# Print columns: user_input, faithfulness, answer_relevancy
df = results.to_pandas()

# Print per-question scores
print(f"{'Question':<50}  {'Faithful':>9}  {'Relevant':>9}")
print("-" * 72)
for _, row in df.iterrows():
    q_short  = str(row.get('user_input', ''))[:48]
    faith    = row.get('faithfulness',    float('nan'))
    relevant = row.get('answer_relevancy', float('nan'))
    print(f"{q_short:<50}  {faith:>9.3f}  {relevant:>9.3f}")

# Print average scores
print("-" * 72)
print(f"{'AVERAGE':<50}  {df['faithfulness'].mean():>9.3f}  {df['answer_relevancy'].mean():>9.3f}")

C:\Users\edgar\AppData\Local\Temp\ipykernel_23440\603712077.py:15: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\edgar\AppData\Local\Temp\ipykernel_23440\603712077.py:15: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\edgar\AppData\Local\Temp\ipykernel_23440\603712077.py:22: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\edgar\AppData\Local\Temp\ipykernel_23440\603712077.py:27: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


Dataset columns : ['user_input', 'response', 'retrieved_contexts']
Dataset rows    : 3

Running RAGAS evaluation (takes ~30–60 seconds)...


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Exception raised in Job[1]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[5]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[3]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})



✅ Evaluation complete!

Question                                             Faithful   Relevant
------------------------------------------------------------------------
What are the main topics covered in this documen        0.462        nan
What is the current price of Bitcoin?                   0.000        nan
What does the document say about deadlines?             1.000        nan
------------------------------------------------------------------------
AVERAGE                                                 0.487        nan


### 📝 Reflection E

1. **Faithfulness score for the out-of-scope question** (Bitcoin price): was it low as expected? What does a low faithfulness score indicate?

   Yes. The Bitcoin question received a faithfulness score of 0.000, the lowest possible score. In general, low faithfulness means the evaluator could not support the response's claims from the retrieved contexts. In this run the application was expected to refuse the out-of-scope question, so the zero must be interpreted cautiously: claim-based metrics can score a refusal or non-answer poorly even when refusing is the correct behavior. The main-topics answer scored 0.462, while the deadlines answer scored 1.000.

2. **chunk_size=300 vs chunk_size=800:** did the RAGAS scores change? Which metric changed more — faithfulness or relevancy? Why do you think that is?

   A valid 300-versus-800 comparison was not executed in E2. The evaluation used only `store_800` with `k=4`, so there is no second set of RAGAS scores from `chunk_size=300`. In addition, every answer-relevancy job failed because Groq rejected RAGAS's request for multiple completions (`n` must be at most 1), leaving all relevancy values as `NaN`. Therefore, the observed results support neither a chunk-size comparison nor a conclusion about which metric changed more. The only valid aggregate reported was average faithfulness of 0.487.

3. **Limitation of this evaluation setup:** we are using an LLM (LLaMA via Groq) as both the RAG generator AND the RAGAS judge. What problem could this cause?

   Using the same LLaMA/Groq model family as both generator and judge can introduce correlated bias or self-preference. The judge may reward wording and reasoning patterns similar to its own, overlook the same kinds of mistakes made during generation, or apply the rubric inconsistently. The failed answer-relevancy calls also show an operational limitation: the evaluator may require API behavior, such as multiple completions, that the selected provider does not support. A stronger evaluation would combine a different judge model with human-reviewed examples and deterministic checks.

4. **In a production RAG system**, how often would you run RAGAS evaluation — and on how many questions?

   In production, I would run a small regression evaluation on every material change to prompts, chunking, embeddings, retrieval settings, models, or source data, and schedule a larger evaluation periodically, such as weekly or before each release. The set should contain hundreds of representative questions for an initial system and grow to thousands for a mature, high-traffic product, including in-scope, out-of-scope, ambiguous, adversarial, and recently failed real queries. Automated scores should be monitored over time and supplemented with sampled human review.

---
## 🧹 Cleanup

Run this cell when you are done to remove all temporary ChromaDB index folders.

In [35]:
dirs_to_clean = [
    "./chroma_lab_notebook",
    "./chroma_exp_300",
    "./chroma_exp_800",
    "./chroma_exp_1500",
    "./chroma_k_exp",
]
for d in dirs_to_clean:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"🗑  Removed {d}")
print("✅ Cleanup done")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: './chroma_lab_notebook\\88baf782-df3b-49b1-9adf-d1ebe6946ade\\data_level0.bin'